# Multi-label klasifikacija torakalnih bolesti

Ovaj projekat se bavi razvojem modela za automatsku detekciju patoloških stanja na osnovu rendgenskih snimaka grudnog koša. Problem je definisan kao multi-label klasifikacija, što znači da jedan snimak može istovremeno sadržavati nula, jednu ili više različitih dijagnoza. 

Fokus je na 14 specifičnih patologija:
Atelectasis, Cardiomegaly, Edema, Effusion, Emphysema, Fibrosis, Hernia, Infiltration, Mass, Nodule, Pleural Thickening, Pneumonia, Pneumothorax i Consolidation.

Rendgenski snimak grudnog koša je jedna od najčešćih dijagnostičkih metoda u medicini. Interpretacija ovih snimaka je lako podložna greškama uslijed različitih faktora. Automatizacija ovog procesa pomoću dubokog učenja nudi bržu identifikacija kritičnih stanja i analizu zasnovanu na hiljadama prethodnih primjera.

U izgradnji modela se koristi **NIH Chest X-ray** dataset, koji sadrži preko 112,000 snimaka toraksa, od preko 30 000 pacijenata.

Za izgradnju modela korišćena je **DenseNet-121** arhitektura.

## DenseNet-121

**DenseNet-121** je vrsta duboke neuronske mreže kod koje je svaki sloj povezan sa svim prethodnim slojevima unutar bloka. 

Za razliku od klasičnih CNN-ova, ovde svaki sloj dobija originalni ulaz, izlaz prvog sloja, izlaz drugog sloja, ..., i tako sve do sloja koji mu prethodi. To znači da informacije stalno cirkulišu kroz mrežu i ponovo se koriste. 

Takva struktura omogućava:
- bolji protok gradijenta (smanjuje problem nestajanja gradijenta tokom treninga) 
- ponovnu upotrebu feature-a (karakteristike koje su već izvučene ne moraju ponovo da se uče). 

Ova arhitektura izabrana je upravo zbog te mogućnosti ponovne upotrebe relevantnih karakteristika, stabilnog protoka gradijenta, ali i zbog činjenice da se pokazao efikasnim upravo za dijagnostifikovanje bolesti na osnovu medicinskih snimaka.

## Izrada modela:

Na početku učitavamo neophodne alate za rad sa podacima, slikama i modelom. 

Pandas i NmuPy za manipulaciju podacima.

TensorFlow i Keras za definisanje, treniranje i evaluaciju modela dubokog učenja.

Scikit-learn za dijeljenje podataka.

ImageDataGenerator za data augmentation i efikasno učitavanje slika sa diska u memoriju GPU tokom treninga.

DenseNet121 arhitektura koju koristimo kao osnovu modela.

Callbacks mehanizmi koji prate proces treniranja.

In [ ]:
import pandas as pd
import numpy as np
import os
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

Prvi konkretan korak jeste priprema podataka.

Prvo se definiše putanja do csv fajla koji sadži metapodatke o slikama. Kolona Finding Labels sadrži dijagnoze u tekstualnom obliku, pri čemu jedna slika može imati više bolesti odjednom. 

Kako bi se podaci prilagodili neuronskoj mreži, prvo se izdvajaju sve jedinstvene bolesti iz ove kolone. Nakon toga se uklanja oznaka No Finding jer ona ne predstavlja patologiju, već odsustvo iste. 

Zatim se vrši transformacija tekstualnih oznaka u numerički format primjenom takozvanog multi-hot encoding-a. Za svaku bolest se kreira posebna binarna kolona u DataFrame-u. Ukoliko je određena bolest prisutna na slici, vrijednost u toj koloni dobija vrijednost 1, dok u suprotnom iznosi 0. 

Nakon obrade oznaka, sledeći korak podrazumijeva povezivanje metapodataka sa stvarnim slikama. 

Na kraju, kompletan skup podataka se dijeli na trening, validacioni i test skup. Trening skup čini 70% podataka i koristi se za učenje modela, dok se preostalih 30% dijeli ravnomijerno na validacioni i test skup. 

Ovim postupkom su podaci transformisani u numerički i organizovan oblik pogodan za treniranje duboke neuronske mreže.

In [ ]:
base_path = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
csv_path = base_path + "/Data_Entry_2017.csv"

df = pd.read_csv(csv_path)

all_labels = (
    df["Finding Labels"]
    .str.split("|")
    .explode()
    .unique()
)

all_labels = sorted([label for label in all_labels if label != "No Finding"])  

for label in all_labels:
    df[label] = df["Finding Labels"].apply(lambda x: 1 if label in x else 0)

image_paths = {}
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".png"):
            image_paths[file] = os.path.join(root, file)

df["path"] = df["Image Index"].map(image_paths)
df = df.dropna(subset=["path"])

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

Nakon pripreme i podjele podataka, sledeći korak u izgradnji modela odnosi se na rješavanje problema neravnoteže klasa (class imbalance). U NIH Chest X-ray skupu podataka pojedine bolesti su znatno rjeđe zastupljene u odnosu na druge. Ukoliko bi se model trenirao bez dodatnih korekcija, imao bi tendenciju da favorizuje češće bolesti, dok bi performanse na rijetkim patologijama bile znatno slabije.

Kako bi se ovaj problem ublažio, izračunavaju se težine za svaku klasu na osnovu odnosa broja negativnih i pozitivnih uzoraka u trening skupu. Za svaku bolest određuje se broj pozitivnih primjera (slika na kojima je bolest prisutna) i broj negativnih primjera (slika bez te bolesti). Težina klase se zatim definiše kao odnos broja negativnih i pozitivnih uzoraka. Na taj način rjeđe bolesti dobijaju veću težinu, dok češće bolesti dobijaju manju težinu. Dodavanje male konstante (1e-5) u imenilac sprječava dijeljenje nulom u slučaju ekstremno rijetkih klasa.

Izračunate težine se zatim konvertuju u TensorFlow tenzor kako bi mogle biti korišćene tokom optimizacije modela.

Na osnovu ovih težina se definiše prilagođena funkcija gubitka - weighted binary crossentropy. Budući da je riječ o multi-label problemu, koristi se binarna entropija za svaku klasu pojedinačno, umjesto softmax funkcije. Ako bi neka klasa imala znatno više primjera od druge, model bi mogao da ignoriše tu klasu sa manje primjera i svejedno imao dobar ukupni rezultat. Ali korišćenjem ove funkcije model je primoran da obrati pažnju na rijetke patologije. Dakle ako model pogriješi kod rijetke bolesti dobija veću kaznu. 

$$BCE = -[y \log(p) + (1 - y) \log(1 - p)]$$

$$WBCE = -[w \cdot y \log(p) + (1 - y) \log(1 - p)]$$

| Simbol | Naziv | Opis |
| :---: | :--- | :--- |
| $y$ | **Stvarna vrijednost** | Labela iz skupa podataka (0 ili 1). |
| $p$ | **Predikcija modela** | Vjerovatnoća koju model predviđa. |
| $w$ | **Težinski faktor** | Izračunata težina: $\frac{\text{broj negativnih}}{\text{broj pozitivnih}}$. |

In [ ]:
pos_weights = []
for label in all_labels:
    pos = train_df[label].sum()
    neg = len(train_df) - pos
    pos_weights.append(neg / (pos + 1e-5))

pos_weights_tensor = tf.constant(np.array(pos_weights), dtype=tf.float32)

def weighted_binary_crossentropy(y_true, y_pred):
    epsilon = 1e-7
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
    loss = - (pos_weights_tensor * y_true * tf.math.log(y_pred) +
              (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

Sledeći korak u izgradnji modela odnosi se na pripremu slika za treniranje neuronske mreže. Budući da se koristi DenseNet-121 arhitektura u okviru transfer learning pristupa, ulazne slike je potrebno prilagoditi standardnoj dimenziji koju model očekuje. U tom smislu definiše se veličina slike od 224×224 piksela, što je standardna ulazna rezolucija za modele trenirane na ImageNet datasetu. Takođe se definiše veličina batch-a od 32 slike, što predstavlja broj uzoraka koji se istovremeno obrađuju tokom jednog koraka optimizacije.

Kako bi se poboljšala generalizacija modela i smanjio rizik od overfitting-a, primjenjuje se data augmentation nad trening skupom. U tu svrhu koristi se ImageDataGenerator, koji omogućava dinamičku transformaciju slika tokom treninga. Sve slike se prvo skaliraju dijeljenjem vrijednosti piksela sa 255, čime se normalizuju u opseg [0,1]. Pored toga, primjenjuju se blage geometrijske transformacije: rotacija do 10 stepeni, horizontalno i vertikalno pomijeranje do 5%, zumiranje do 10%, kao i horizontalno preslikavanje (horizontal flip). Ove transformacije simuliraju različite varijacije u položaju i orijentaciji pacijenta, čime se model čini robusnijim na male promjene u ulaznim podacima.

Važno je naglasiti da se augmentacija primjenjuje isključivo nad trening skupom. Validacioni skup prolazi samo kroz proces skaliranja, bez dodatnih transformacija, kako bi evaluacija performansi modela bila realistična i odražavala stvarne podatke.

Funkcija flow_from_dataframe čita slike sa diska, prilagođava ih, uzima njihove labele iz DataFrame-a i vraća ih u batch-evima. Znači, umjesto da učitamo sve slike u RAM, generator ih učitava postepeno, tokom treninga.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="path",
    y_col=all_labels,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw"
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="path",
    y_col=all_labels,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw"
)

Kada je gotova priprema podataka, prelazimo na konstrukciju modela zasnovanog na transfer learning pristupu. Kao osnovna arhitektura koristi se DenseNet-121, prethodno trenirana na ImageNet datasetu. Korišćenjem parametra **weights="imagenet"** model preuzima unaprijed naučene težine, što omogućava efikasnije učenje i bržu konvergenciju, posebno u situacijama kada je količina specifičnih medicinskih podataka ograničena. 

Prilikom inicijalizacije modela postavlja se **include_top=False**, čime se uklanja originalni klasifikacioni sloj DenseNet-121 arhitekture koji je namijenjen ImageNet klasifikaciji sa 1000 klasa. Na taj način zadržava se samo konvoluciona baza modela, odnosno dio mreže zadužen za ekstrakciju vizuelnih karakteristika. Ulazna dimenzija se definiše kao 224×224×3, što odgovara standardnoj veličini RGB slika korišćenoj u prethodno treniranom modelu.

Nakon inicijalizacije DenseNet-121 konvolucione baze, njen izlaz se koristi kao ulaz u završni dio modela. Izlaz baznog modela predstavlja skup dubokih feature mapa.

Na ove feature mape se primjenjuje Global Average Pooling sloj, čime se svaka mapa redukuje na jednu prosječnu vrijednost. Ovim postupkom se smanjuje dimenzionalnost podataka i dobija kompaktan vektor karakteristika. Prednost ovog pristupa je manji broj parametara i manji rizik od overfitting-a u poređenju sa klasičnim potpuno povezanim slojevima.

Dobijeni vektor se zatim prosljeđuje u završni Dense sloj sa brojem neurona jednakim broju bolesti u datasetu. Kao aktivaciona funkcija koristi se sigmoid, jer omogućava nezavisno predviđanje svake klase, što je neophodno u multi-label klasifikaciji.

Na kraju se formira kompletan model povezivanjem ulaza bazne DenseNet-121 mreže sa novodefinisanim izlaznim slojem, čime se dobija arhitektura prilagođena detekciji više torakalnih bolesti sa jedne slike.


In [ ]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
output = Dense(len(all_labels), activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

Da bi se proces treniranja modela učinio efikasnijim i stabilnijim, koriste se tzv. callbacks – funkcije koje prate i automatski upravljaju različitim aspektima treninga. U ovom projektu definisane su tri ključne strategije:

- **ModelCheckpoint**

    Ova funkcija čuva najbolje stanje modela tokom treninga u fajl "best_model.h5". Praćenje se vrši prema metrikama validacionog skupa, u ovom slučaju val_auc, što omogućava da se sačuva model koji najbolje prepoznaje bolesti, čak i ako se performanse na trening skupu dalje pogoršavaju. Parametar save_best_only=True garantuje da se čuva samo najbolja verzija modela, čime se izbjegava nepotrebno skladištenje.

- **EarlyStopping** 

    Early stopping prati promjene u validacionom AUC-u i prekida trening ako performanse ne napreduju određeni broj epoha (patience=3). Na ovaj način se sprječava pretreniranje (overfitting) i štedi vrijeme i resursi, jer model neće dalje učiti kada se već postigne optimalna tačka.

- **ReduceLROnPlateau** 

    Ova strategija automatski smanjuje brzinu učenja (learning rate) ako validacioni gubitak (val_loss) prestane da opada tokom definisanog broja epoha (patience=2). Smanjenje learning rate-a za faktor 0.1 omogućava finije podešavanje težina modela u kasnijim fazama treninga, što često poboljšava konačne performanse. Parametar min_lr=1e-7 postavlja donju granicu kako se learning rate ne bi smanjivao na previše malu vrijednost.

In [ ]:
callbacks = [
    ModelCheckpoint("best_model.h5", monitor="val_auc",
                    mode="max", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_auc", mode="max",
                  patience=3, verbose=1),
    ReduceLROnPlateau(monitor="val_loss",
                      factor=0.1, patience=2,
                      min_lr=1e-7, verbose=1)
]

U prvoj fazi treniranja modela koristi se pristup poznat kao **frozen backbone**. To znači da su svi slojevi bazne DenseNet-121 mreže zamrznuti, odnosno njihove težine se ne mijenjaju tokom početnog treninga.

Cilj ovog pristupa je da se zadrže prethodno naučene vizuelne karakteristike DenseNet-121, koje su stečene na ImageNet datasetu, i da se samo novi dodati slojevi (Global Average Pooling i Dense sloj za multi-label klasifikaciju) nauče da povežu te karakteristike sa specifičnim torakalnim bolestima. Na taj način se smanjuje rizik od overfitting-a, posebno jer medicinski dataset može biti znatno manji od ImageNet-a.

Model se kompajlira pomoću **Adam** optimizatora sa learning rate-om 1e-4, koji je pogodna vrijednost za početno učenje novih slojeva. Kao funkcija gubitka koristi se prethodno definisana weighted binary cross-entropy, koja uzima u obzir neravnotežu klasa. Metričke performanse modela se prate preko AUC-a za multi-label klasifikaciju i binary accuracy, što omogućava ocjenu kako model detektuje pojedinačne bolesti i koliko je tačan po svakoj klasi.

Trening se vrši pomoću **model.fit** funkcije. Model se trenira 7 epoha, što predstavlja dovoljno vremena da novi slojevi nauče da povežu ekstraktovane feature mape sa tačnim bolestima, a da se pri tom ne mijenja težina bazne DenseNet-121 mreže.

In [ ]:
#FAZA 1: Frozen backbone

for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=weighted_binary_crossentropy,
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc"),
             "binary_accuracy"]
)

history_frozen = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=7,
    callbacks=callbacks
)

Nakon inicijalnog treniranja sa zamrznutom baznom DenseNet-121 mrežom, sledeća faza koristi tehniku fine-tuninga. Cilj ove faze je dodatno prilagoditi model specifičnom datasetu X-ray slika i poboljšati performanse, posebno u prepoznavanju suptilnih obrazaca bolesti.

U ovoj fazi se trenira samo posljednjih 60 slojeva baznog modela, dok su ostali slojevi i dalje zamrznuti. 
- Otključava se posljednjih 60 slojeva koji uče složenije i apstraktnije karakteristike, bliže kraju mreže, koje su najviše specifične za dataset.
- Rani i srednji slojevi ostaju zamrznuti jer uče opšte vizuelne obrasce (ivice, teksture, oblike) koji su korisni i za X-ray slike.

Model se ponovo kompajlira, ali ovog puta sa manjim learning rate-om 1e-5, jer su težine baznog modela već naučene i male promjene su dovoljne da se fino prilagode novom datasetu.

Trening se vrši tokom osam epoha.

In [ ]:
#FAZA 2: Fine-tuning posljednjih 60 slojeva 

for layer in base_model.layers[-60:]:
    layer.trainable = True

for layer in base_model.layers[:-60]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # manji LR
    loss=weighted_binary_crossentropy,
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="auc"),
             "binary_accuracy"]
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=8,
    callbacks=callbacks
)

## Trening:

Nakon što je model treniran i sačuvan u fajl "best_model.h5", sledeći korak je provjera performansi na test skupu, koji sadrži slike koje model ranije nije vidio. 

Za treniranje se koristi generator koji učita slike iz test DateFrame-a. Slično kao i validacioni generator, slike se skaliraju, ali se ne primjenjuje augmentacija, kako bi evaluacija održavala stvarne performanse modela. 

Ovim postupkom se omogućava nepristrasna procjena tačnosti modela, uključujući metrike kao što su AUC, precision-recall krive i klasifikacioni izveštaj.

Ova faza je ključna jer pokazuje koliko model zaista može da prepozna torakalne bolesti na novim, neviđenim X-ray snimcima i daje osnovu za procjenu njegovih praktičnih performansi u realnim uslovima.

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('/kaggle/input/datasets/brankakovacevic/best-model/best_model.h5', custom_objects={'weighted_binary_crossentropy': weighted_binary_crossentropy})

# 2. Test generator (SHUFFLE FALSE)
test_generator = val_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="path",
    y_col=all_labels,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False
)

predictions = model.predict(test_generator, steps=len(test_generator), verbose=1)
test_labels = test_df[all_labels].values
